# SEED-VII EEGNet × LoRA-LLM — Kaggle Pipeline

本 Notebook 在 **Kaggle** 环境中运行完整 Pipeline：

1. 从 GitHub 克隆仓库并安装依赖
2. 通过 ModelScope Dataset API 拉取 `DEREKVERSE/SEED-VII` 数据集
3. NPZ 预处理（已完成则跳过）
4. 下载/加载 LLM 模型
5. 训练对比学习双塔
6. EEG 编码推理

> **设计原则**：本 Notebook 不将代码打包进数据集，而是直接 `git clone` 关联 GitHub 仓库，
> 所有中间产物（数据集、NPZ、模型权重）均落盘到 Kaggle 工作目录，若已存在则跳过对应步骤。

## 0. 环境准备：克隆仓库 + 安装依赖

In [ ]:
import sys
from pathlib import Path

print('Python:', sys.version)

# ── Kaggle 工作目录 ──
WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)

# ── 克隆仓库（如果尚未克隆）──
REPO = WORK / 'EEG_OPUS'
if not (REPO / 'seedvii_modal_contrastive_lora' / 'pyproject.toml').exists():
    !git clone https://github.com/PRIMOCOSMOS/EEG_OPUS.git {REPO}
else:
    print(f'Repo already cloned at {REPO}')

PROJECT = REPO / 'seedvii_modal_contrastive_lora'
assert (PROJECT / 'pyproject.toml').exists(), f'Repository structure error: {PROJECT}'
print('PROJECT:', PROJECT)

In [ ]:
# ── 安装依赖 ──
%pip install -q -r {PROJECT / 'requirements.txt'}
%pip install -q modelscope  # Kaggle 默认不含 modelscope
%pip install -q -e {PROJECT}

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

print('[OK] Dependencies installed')

## 1. 路径配置

In [ ]:
import os
import yaml
from pathlib import Path

# ── 可自定义路径 ──
DATASET_ID       = 'DEREKVERSE/SEED-VII'
LOCAL_DATASET_DIR = WORK / 'seedvii_ms_dataset'       # 原始 .mat + .csv
NPZ_DIR           = WORK / 'seedvii_npz'               # 预处理后的 NPZ shard
RUN_DIR           = WORK / 'seedvii_contrastive_runs' / 'run_valence3'
MODEL_DIR         = WORK / 'models' / 'Qwen2.5-0.5B-Instruct'

for d in [LOCAL_DATASET_DIR, NPZ_DIR, RUN_DIR, MODEL_DIR.parent]:
    d.mkdir(parents=True, exist_ok=True)

print(f'LOCAL_DATASET_DIR = {LOCAL_DATASET_DIR}')
print(f'NPZ_DIR           = {NPZ_DIR}')
print(f'RUN_DIR           = {RUN_DIR}')
print(f'MODEL_DIR         = {MODEL_DIR}')

## 2. 通过 ModelScope Dataset API 拉取数据集

拉取 `1-20.mat` 和 `text_protocol.csv`。已存在则跳过。

In [ ]:
from seedvii_contrastive.scripts.download_modelscope_seedvii import find_downloaded_paths
from seedvii_contrastive.data.discovery import SUBJECT_FILE_NAMES

# 检查是否已经下载完成
already_downloaded = False
if LOCAL_DATASET_DIR.exists():
    eeg_root_test, text_csv_test = find_downloaded_paths(LOCAL_DATASET_DIR)
    if eeg_root_test is not None and text_csv_test is not None:
        mats = sorted([p.name for p in Path(eeg_root_test).glob('*.mat') 
                       if p.name in SUBJECT_FILE_NAMES])
        if len(mats) >= 20:
            already_downloaded = True
            print(f'[SKIP] Dataset already downloaded: {len(mats)} subject .mat files found')

if not already_downloaded:
    # 如果是私有数据集需设置 MODELSCOPE_TOKEN 环境变量
    # os.environ['MODELSCOPE_TOKEN'] = 'your-token'
    !python -m seedvii_contrastive.scripts.download_modelscope_seedvii \
        --dataset-id {DATASET_ID} \
        --local-dir {LOCAL_DATASET_DIR} \
        --max-workers 4

In [ ]:
# ── 自动发现下载后的 EEG_ROOT 和 TEXT_CSV ──
EEG_ROOT, TEXT_CSV = find_downloaded_paths(LOCAL_DATASET_DIR)
print(f'EEG_ROOT = {EEG_ROOT}')
print(f'TEXT_CSV = {TEXT_CSV}')
assert EEG_ROOT is not None, '未找到 1-20.mat 所在目录'
assert TEXT_CSV is not None, '未找到 text_protocol*.csv；LLM Tower 必须使用 L2 文本协议'

## 3. 下载 / 加载 LLM 模型

推荐 `Qwen/Qwen2.5-0.5B-Instruct`。若已存在则跳过。

In [ ]:
if not (MODEL_DIR / 'config.json').exists():
    from modelscope import snapshot_download
    model_path = snapshot_download('Qwen/Qwen2.5-0.5B-Instruct', 
                                   cache_dir=str(WORK / 'models'))
    print(f'Downloaded LLM to: {model_path}')
    # 如果返回路径与 MODEL_DIR 不同，做符号链接或更新变量
    if Path(model_path) != MODEL_DIR:
        MODEL_DIR = Path(model_path)
else:
    print(f'[SKIP] LLM model already exists: {MODEL_DIR}')

## 4. NPZ 预处理

每个 subject/trial 独立处理：中间 60%，4s non-overlap windows，最多每 clip 采样固定窗口数。
**若 `index.csv` 已存在则跳过。**

In [ ]:
if not (NPZ_DIR / 'index.csv').exists():
    !python -m seedvii_contrastive.scripts.preprocess_npz \
        --input-root {EEG_ROOT} \
        --output-dir {NPZ_DIR} \
        --subjects 1-20 \
        --window-sec 4 --stride-sec 4 \
        --center-ratio 0.60 \
        --max-windows-per-clip 12 \
        --shard-size 512
else:
    print(f'[SKIP] NPZ index already exists: {NPZ_DIR / "index.csv"}')

## 5. 写入运行配置

In [ ]:
# ── 从基础配置模板出发，注入运行时路径 ──
base_cfg_path = PROJECT / 'configs' / 'modelscope_default.yaml'
cfg = yaml.safe_load(open(base_cfg_path, 'r', encoding='utf-8'))

cfg['data']['modelscope_dataset_id'] = DATASET_ID
cfg['data']['local_dataset_dir']     = str(LOCAL_DATASET_DIR)
cfg['data']['eeg_root']              = str(EEG_ROOT)
cfg['data']['text_csv_path']         = str(TEXT_CSV)
cfg['data']['npz_dir']               = str(NPZ_DIR)
cfg['runtime']['output_dir']         = str(RUN_DIR)
cfg['model']['llm']['model_name_or_path'] = str(MODEL_DIR)

# ── Kaggle GPU 适配：T4 显存约 15GB，降低 batch_size ──
# 如果你使用 P100 (16GB) 或双 T4，可调大
cfg['train']['batch_size'] = 48           # 原默认 96，Kaggle T4 减半
cfg['model']['llm']['gradient_checkpointing'] = True

# ── 保存运行配置 ──
run_cfg = RUN_DIR / 'config.yaml'
RUN_DIR.mkdir(parents=True, exist_ok=True)
yaml.safe_dump(cfg, open(run_cfg, 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
print(open(run_cfg, 'r', encoding='utf-8').read())

## 6. 训练

`train.resume=true` 时自动从 `last.pt` 继续训练；验证集 macro-F1 最优保存 `best.pt`。

In [ ]:
!python -m seedvii_contrastive.scripts.train_contrastive --config {run_cfg}

## 7. EEG 编码 / 推理

使用训练好的最佳模型对验证集进行编码，输出 embedding 和预测结果。

In [ ]:
BEST_CKPT = RUN_DIR / 'best.pt'
OUT_EMB   = RUN_DIR / 'val_embeddings.npz'

if BEST_CKPT.exists():
    !python -m seedvii_contrastive.scripts.encode_eeg \
        --config {run_cfg} \
        --checkpoint {BEST_CKPT} \
        --split val \
        --out {OUT_EMB}
    print(f'[OK] Embeddings saved to: {OUT_EMB}')
else:
    print(f'[WARN] Best checkpoint not found: {BEST_CKPT}')
    print('Training may not have completed successfully.')

## 8. 训练曲线（可选）

In [ ]:
# ── 简易可视化：如果训练日志存在，绘制 loss 曲线 ──
import matplotlib.pyplot as plt
import json

log_path = RUN_DIR / 'metrics.jsonl'
if log_path.exists():
    import pandas as pd
    df = pd.read_json(log_path, lines=True)
    if 'loss' in df.columns:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        axes[0].plot(df['loss'].dropna().values, alpha=0.7)
        axes[0].set_title('Training Loss')
        axes[0].set_xlabel('Step')
        if 'val_macro_f1' in df.columns:
            val = df['val_macro_f1'].dropna()
            axes[1].plot(val.values, marker='o')
            axes[1].set_title('Validation Macro-F1')
            axes[1].set_xlabel('Epoch')
        plt.tight_layout()
        plt.show()
else:
    print('No metrics log found.')

---

## Kaggle 注意事项

- **GPU**：Kaggle 提供 T4 ×2（约 15GB 单卡）或 P100（16GB），batch_size 已调整至 48，显存不足可进一步减小。
- **时间限制**：Kaggle Notebook 单次 Session 最多运行 9 小时，完整训练约需数小时。建议开启 `resume=true` 以便中断后继续。
- **持久化**：`/kaggle/working` 下的文件在 Session 结束后保留为 Notebook Output；下载的数据集和模型可作为 Kaggle Dataset 持久化以便复用。
- **网络**：Kaggle 有外网访问权限，ModelScope API 可直接使用。